# Bidirectional Sleep ↔ Activity Analysis

**Direction 1:** Does previous-night sleep quality predict next-day step count?  
**Direction 2:** Do high-activity days predict worse/better next-night recovery (resting HR, sleep duration, fragmentation)?

All physiological metrics are **within-patient z-scored** to capture deviations from each person's own baseline.  

In [1]:
import sys, os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import spearmanr
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

PROJECT_ROOT = os.path.abspath(
    os.path.join(os.getcwd(), '..') if os.getcwd().endswith('notebooks') else os.getcwd()
)
sys.path.insert(0, PROJECT_ROOT)
import data_prep as dp

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
FIG_DIR  = os.path.join(PROJECT_ROOT, 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

DISEASE_ORDER  = ['Early Disease Stage', 'Fast Disease Progression', 'Late Disease Stage']
DISEASE_COLORS = {
    'Early Disease Stage':      '#4C72B0',
    'Fast Disease Progression': '#DD8452',
    'Late Disease Stage':       '#55A868',
}

print('Setup complete.')

Setup complete.


## Data Loading

In [5]:
import glob

# ── Hourly sensor data ────────────────────────────────────────────────────────
files  = glob.glob(os.path.join(DATA_DIR, 'Hourly Sensor Data', 'RHourly_*.csv'))
chunks = []
for fp in files:
    pid = int(os.path.basename(fp).replace('RHourly_', '').replace('.csv', ''))
    c = pd.read_csv(fp); c['id'] = pid; chunks.append(c)
sensor = pd.concat(chunks, ignore_index=True)
sensor['time']      = pd.to_datetime(sensor['time'])
sensor['heartrate'] = pd.to_numeric(sensor['heartrate'], errors='coerce')

# ── Clinical markers ─────────────────────────────────────────────────────────
clinical = pd.read_csv(os.path.join(DATA_DIR, 'ClinicalMarkers_final.csv'))
clinical.columns = clinical.columns.str.strip().str.lower().str.replace('.', '_', regex=False)
clinical['disease_type'] = clinical['disease_type'].str.strip()
df_merged = clinical[['id', 'age', 'sex', 'disease_type']].merge(sensor, on='id', how='left')

# ── Preceding-night load: sum_sleep_minute on day D = sleep from [18:00 D-1, 06:00 D]
df    = dp.get_complete_patient_days(df_merged, sleep_shift_hour=6, complete=True)
daily = dp.aggregate_to_daily(df)
daily = daily[daily['sum_sleep_minute'] > 0].copy()

# ── Next-night load: sum_sleep_minute on day D = sleep from [18:00 D, 06:00 D+1]
df_next    = dp.get_complete_patient_days(df_merged, sleep_shift_hour=-18, complete=True)
daily_next = dp.aggregate_to_daily(df_next)
daily_next = daily_next[daily_next['sum_sleep_minute'] > 0].copy()

daily['age_years']  = 2026 - daily['age']
daily['sex_binary'] = (daily['sex'] == 'Male').astype(int)
daily['disease_type'] = pd.Categorical(daily['disease_type'], categories=DISEASE_ORDER, ordered=True)
daily_next['age_years']  = 2026 - daily_next['age']
daily_next['sex_binary'] = (daily_next['sex'] == 'Male').astype(int)
daily_next['disease_type'] = pd.Categorical(daily_next['disease_type'], categories=DISEASE_ORDER, ordered=True)

daily = daily.sort_values(['id', 'date']).reset_index(drop=True)
daily_next = daily_next.sort_values(['id', 'date']).reset_index(drop=True)

print(f'Daily rows: {len(daily):,}  |  patients: {daily["id"].nunique()}')
print(f'Date range: {daily["date"].min().date()} → {daily["date"].max().date()}')
print(daily['disease_type'].value_counts())
print()
print(f'Daily NEXT rows: {len(daily_next):,}  |  patients: {daily_next["id"].nunique()}')
print(f'Date range: {daily_next["date"].min().date()} → {daily_next["date"].max().date()}')
print(daily_next['disease_type'].value_counts())

Computing per-day features...
Computing per-day features...
Daily rows: 1,861  |  patients: 43
Date range: 2021-01-09 → 2021-11-14
disease_type
Late Disease Stage          887
Early Disease Stage         672
Fast Disease Progression    302
Name: count, dtype: int64

Daily NEXT rows: 1,733  |  patients: 42
Date range: 2021-01-09 → 2021-11-13
disease_type
Late Disease Stage          829
Early Disease Stage         625
Fast Disease Progression    279
Name: count, dtype: int64


## Feature Engineering

In [ ]:
# check patient day alignment
_pid = daily['id'].iloc[100]
_pat = daily[daily['id'] == _pid].head(4)
_pat[['date', 'sum_sleep_minute', 'sleep_fragmentation_min', 'resting_hr']]

,date,sum_sleep_minute,sleep_fragmentation_min,resting_hr
99,2021-09-03,499.0,41.0,67.792700
100,2021-09-06,514.0,26.0,66.453246
101,2021-09-07,436.0,104.0,65.996908
102,2021-09-08,512.0,28.0,66.976511


In [ ]:
# check patient day alignment for next-night load
_pat_next = daily_next[daily_next['id'] == _pid].head(4)
_pat_next[['date', 'sum_sleep_minute', 'sleep_fragmentation_min', 'resting_hr']]

,date,sum_sleep_minute,sleep_fragmentation_min,resting_hr
95,2021-09-03,458.0,22.0,63.487878
96,2021-09-06,436.0,104.0,65.996908
97,2021-09-07,512.0,28.0,66.976511
98,2021-09-08,543.0,57.0,67.169872


In [11]:
# ── Within-patient z-scores ───────────────────────────────────────────────────
Z_COLS = [
    'daily_steps', 'sum_sleep_minute', 'sleep_fragmentation_min', 'resting_hr',
]

def _patient_zscore(x):
    s = x.std()
    return (x - x.mean()) / s if s > 0 else pd.Series(0.0, index=x.index)

for col in Z_COLS:
    if col in daily.columns:
        daily[col + '_z'] = daily.groupby('id')[col].transform(_patient_zscore)

for col in Z_COLS:
    if col in daily_next.columns:
        daily_next[col + '_z'] = daily_next.groupby('id')[col].transform(_patient_zscore)

print('Z-score check (should be ~0 mean, ~1 std within each patient):')
print(daily.groupby('id')['daily_steps_z'].agg(['mean', 'std']).describe().round(3))
print(daily_next.groupby('id')['daily_steps_z'].agg(['mean', 'std']).describe().round(3))

Z-score check (should be ~0 mean, ~1 std within each patient):
       mean   std
count  43.0  42.0
mean   -0.0   1.0
std     0.0   0.0
min    -0.0   1.0
25%    -0.0   1.0
50%    -0.0   1.0
75%     0.0   1.0
max     0.0   1.0
       mean   std
count  42.0  42.0
mean   -0.0   1.0
std     0.0   0.0
min    -0.0   1.0
25%    -0.0   1.0
50%    -0.0   1.0
75%     0.0   1.0
max     0.0   1.0


In [20]:
def _qcut_safe(x, labels):
    try:
        return pd.qcut(x.rank(method='first'), len(labels), labels=labels)
    except ValueError:
        return pd.Series(pd.NA, index=x.index)


def add_quartile_flags(df):
    """Add within-patient activity/sleep quartile labels and high/low flags.

    Works on any aggregate_to_daily output; uses sum_sleep_minute as the sleep
    column (which equals the preceding-night window on daily, next-night on daily_next).
    """
    d = df.copy()

    # ── Quartile labels ──────────────────────────────────────────────────────
    d['steps_q'] = d.groupby('id')['daily_steps'].transform(
        lambda x: _qcut_safe(x, ['Q1 (low)', 'Q2', 'Q3', 'Q4 (high)'])
    )
    d['sleep_q'] = d.groupby('id')['sum_sleep_minute'].transform(
        lambda x: _qcut_safe(x, ['Q1 (poor)', 'Q2', 'Q3', 'Q4 (good)'])
    )

    # ── 75th / 25th percentile flags (for contrast/time-series plots) ────────
    d['steps_p75']    = d.groupby('id')['daily_steps'].transform(lambda x: x.quantile(0.75))
    d['steps_p25']    = d.groupby('id')['daily_steps'].transform(lambda x: x.quantile(0.25))
    d['is_high_step'] = d['daily_steps'] >= d['steps_p75']
    d['is_low_step']  = d['daily_steps'] <= d['steps_p25']

    d['sleep_p75']     = d.groupby('id')['sum_sleep_minute'].transform(lambda x: x.quantile(0.75))
    d['sleep_p25']     = d.groupby('id')['sum_sleep_minute'].transform(lambda x: x.quantile(0.25))
    d['is_good_sleep'] = d['sum_sleep_minute'] >= d['sleep_p75']
    d['is_poor_sleep'] = d['sum_sleep_minute'] <= d['sleep_p25']

    return d


daily      = add_quartile_flags(daily)
daily_next = add_quartile_flags(daily_next)

CUTOFF_COLS = ['steps_p25', 'steps_p75', 'sleep_p25', 'sleep_p75']

for name, d in [('daily', daily), ('daily_next', daily_next)]:
    print(f'=== {name} ===')
    print('  steps_q:', d['steps_q'].value_counts().sort_index().to_dict())
    print('  sleep_q:', d['sleep_q'].value_counts().sort_index().to_dict())
    print(f'  High-step: {d["is_high_step"].sum():,}  |  Low-step: {d["is_low_step"].sum():,}')
    print(f'  Good-sleep: {d["is_good_sleep"].sum():,}  |  Poor-sleep: {d["is_poor_sleep"].sum():,}')

    cutoffs = d.groupby('id')[CUTOFF_COLS].first()
    print(f'\n  Per-patient cutoff values (steps in count, sleep in minutes):')
    print(f'  {"":6s}  {"steps_p25":>10s}  {"steps_p75":>10s}  {"sleep_p25":>10s}  {"sleep_p75":>10s}')
    for pid, row in cutoffs.iterrows():
        print(f'  {pid:6d}  {row["steps_p25"]:10.0f}  {row["steps_p75"]:10.0f}  '
              f'{row["sleep_p25"]:10.0f}  {row["sleep_p75"]:10.0f}')
    print(f'  {"median":6s}  {cutoffs["steps_p25"].median():10.0f}  {cutoffs["steps_p75"].median():10.0f}  '
          f'{cutoffs["sleep_p25"].median():10.0f}  {cutoffs["sleep_p75"].median():10.0f}')
    print()

=== daily ===
  steps_q: {'Q1 (low)': 480, 'Q2': 461, 'Q3': 447, 'Q4 (high)': 472}
  sleep_q: {'Q1 (poor)': 480, 'Q2': 461, 'Q3': 447, 'Q4 (good)': 472}
  High-step: 481  |  Low-step: 481
  Good-sleep: 484  |  Poor-sleep: 486

  Per-patient cutoff values (steps in count, sleep in minutes):
           steps_p25   steps_p75   sleep_p25   sleep_p75
    1120        6027        8234         380         522
    1556       14483       18382         346         462
    1971        2148        3009         436         535
    2130        1737        2700         428         549
    2589        3969        9454         448         553
    2676        8046       12535         394         464
    2815        8787       12243         480         559
    2854       11826       19526         372         523
    2909       11648       17702         406         497
    3389        4354        6769         312         489
    3405        3947        7845         171         255
    4063       12195     

## Direction 1: Sleep → Next-Day Activity

Does the quality of last night's sleep predict how active the patient is today?

In [24]:
# ── Scatter: each previous-night sleep metric vs today's steps_z ──────────────
D1_SLEEP_METRICS = [
    ('sum_sleep_minute_z',   'Prev-night sleep duration (z)',   'More sleep → more steps?'),
    ('sleep_fragmentation_min_z',  'Prev-night fragmentation (z)',    'More fragmentation → fewer steps?'),
    ('resting_hr_z',  'Prev-night resting HR (z)',       'Higher night HR → fewer steps?'),
]

_plot_df = daily.dropna(subset=['daily_steps_z'] + [m[0] for m in D1_SLEEP_METRICS])

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[m[2] for m in D1_SLEEP_METRICS],
    shared_yaxes=True,
)

for col_idx, (x_col, x_label, _) in enumerate(D1_SLEEP_METRICS, start=1):
    for disease in DISEASE_ORDER:
        sub = _plot_df[_plot_df['disease_type'] == disease]
        fig.add_trace(
            go.Scatter(
                x=sub[x_col], y=sub['daily_steps_z'],
                mode='markers',
                marker=dict(color=DISEASE_COLORS[disease], size=4, opacity=0.45),
                name=disease,
                legendgroup=disease,
                showlegend=(col_idx == 1),
            ),
            row=1, col=col_idx,
        )
        # OLS trend line
        _x = sub[x_col].dropna()
        _y = sub.loc[_x.index, 'daily_steps_z']
        if len(_x) > 5:
            _m, _b = np.polyfit(_x, _y, 1)
            _xr = np.array([_x.min(), _x.max()])
            fig.add_trace(
                go.Scatter(
                    x=_xr, y=_m * _xr + _b,
                    mode='lines',
                    line=dict(color=DISEASE_COLORS[disease], width=2),
                    showlegend=False,
                    legendgroup=disease,
                ),
                row=1, col=col_idx,
            )
    fig.update_xaxes(title_text=x_label, row=1, col=col_idx)

fig.update_yaxes(title_text='Today\'s steps (z)', row=1, col=1)
fig.update_layout(
    title='Direction 1: Previous-Night Sleep → Today\'s Steps<br>'
          '<sup>Within-patient z-scores · OLS trend per disease stage</sup>',
    height=480, width=1100,
    legend=dict(orientation='h', yanchor='bottom', y=1.08),
)
fig.show()

In [25]:
# ── Violin: previous-night sleep quality quartile → today's steps_z ───────────
_plot_df = daily.dropna(subset=['sleep_q', 'daily_steps_z', 'disease_type'])
_plot_df['sleep_q'] = _plot_df['sleep_q'].astype(str)
Q_ORDER = ['Q1 (poor)', 'Q2', 'Q3', 'Q4 (good)']

fig = px.violin(
    _plot_df,
    x='sleep_q', y='daily_steps_z',
    color='disease_type',
    category_orders={
        'sleep_q': Q_ORDER,
        'disease_type': DISEASE_ORDER,
    },
    color_discrete_map=DISEASE_COLORS,
    box=True, points=False,
    title='Direction 1: Prev-Night Sleep Quartile → Today\'s Steps (z)<br>'
          '<sup>Q1=shortest sleep, Q4=longest sleep · within-patient quartiles</sup>',
    labels={'sleep_q': 'Prev-night sleep duration quartile', 'daily_steps_z': 'Today\'s steps (z)'},
    height=480, width=900,
)
fig.update_layout(violinmode='group')
fig.show()

In [26]:
# ── Violin: sleep fragmentation quartile → today's steps_z ───────────────────
# Q1=least fragmented (best), Q4=most fragmented (worst)
daily['frag_q'] = daily.groupby('id')['sleep_fragmentation_min'].transform(
    lambda x: _qcut_safe(x, ['Q1 (least frag)', 'Q2', 'Q3', 'Q4 (most frag)'])
)

_plot_df = daily.dropna(subset=['frag_q', 'daily_steps_z', 'disease_type'])
_plot_df['frag_q'] = _plot_df['frag_q'].astype(str)
FRAG_ORDER = ['Q1 (least frag)', 'Q2', 'Q3', 'Q4 (most frag)']

fig = px.violin(
    _plot_df,
    x='frag_q', y='daily_steps_z',
    color='disease_type',
    category_orders={'frag_q': FRAG_ORDER, 'disease_type': DISEASE_ORDER},
    color_discrete_map=DISEASE_COLORS,
    box=True, points=False,
    title='Direction 1: Prev-Night Sleep Fragmentation Quartile → Today\'s Steps (z)<br>'
          '<sup>Q4 = most fragmented/restless night · within-patient quartiles</sup>',
    labels={'frag_q': 'Prev-night fragmentation quartile', 'daily_steps_z': 'Today\'s steps (z)'},
    height=480, width=900,
)
fig.update_layout(violinmode='group')
fig.show()

In [49]:
# ── Per-patient Spearman r: sleep metrics → daily_steps_z ─────────────────────
D1_PAIRS = [
    ('resting_hr',               'Prev resting\nHR'),
    ('sum_sleep_minute',         'Prev sleep\nduration'),
    ('sleep_fragmentation_min',  'Prev sleep\nfragmentation'),
]

rows_d1 = []
for pid in sorted(daily['id'].unique()):
    pat = daily[daily['id'] == pid]
    disease = pat['disease_type'].iloc[0]
    n       = len(pat)
    row     = {'id': pid, 'disease_type': disease, 'n': n}
    for src_col, label in D1_PAIRS:
        sub = pat[['daily_steps_z', src_col]].dropna()
        if len(sub) >= 10:
            r, p = spearmanr(sub[src_col], sub['daily_steps_z'])
        else:
            r, p = np.nan, np.nan
        row[f'r_{src_col}']   = r
        row[f'sig_{src_col}'] = '*' if (not np.isnan(p) and p < 0.05) else ''
    rows_d1.append(row)

corr_d1 = pd.DataFrame(rows_d1)
corr_d1['disease_ord'] = corr_d1['disease_type'].map(
    {d: i for i, d in enumerate(DISEASE_ORDER)}
)
corr_d1 = corr_d1.sort_values(['disease_ord', 'r_resting_hr'], ascending=[True, False])

r_cols     = [f'r_{c}' for c, _ in D1_PAIRS]
labels     = [l for _, l in D1_PAIRS]
r_matrix   = corr_d1[r_cols].values
sig_matrix = corr_d1[[f'sig_{c}' for c, _ in D1_PAIRS]].values

y_labels = [
    f"{row['id']} ({str(row['disease_type'])[:5]})"
    for _, row in corr_d1.iterrows()
]

fig = px.imshow(
    r_matrix,
    x=labels,
    y=y_labels,
    color_continuous_scale='RdBu_r',
    color_continuous_midpoint=0,
    range_color=[-1, 1],
    title='Direction 1: Per-Patient Spearman r (Sleep → Steps)<br>'
          '<sup>* p < 0.05 · sorted by disease stage · red = positive, blue = negative</sup>',
    aspect='auto',
    height=max(500, len(corr_d1) * 20),
    width=550,
)
for i in range(r_matrix.shape[0]):
    for j in range(r_matrix.shape[1]):
        if sig_matrix[i, j] == '*':
            fig.add_annotation(x=j, y=i, text='*', showarrow=False,
                               font=dict(size=14, color='black'))
fig.update_layout(coloraxis_colorbar=dict(title='Spearman r'))
fig.update_yaxes(tickmode='linear', dtick=1, tickfont=dict(size=9))
fig.show()

print('\nMedian r across patients:')
for src_col, label in D1_PAIRS:
    med   = corr_d1[f'r_{src_col}'].median()
    n_sig = (corr_d1[f'sig_{src_col}'] == '*').sum()
    print(f'  {label.replace(chr(10)," "):30s}  median r={med:+.3f}  significant in {n_sig}/{len(corr_d1)} patients')


Median r across patients:
  Prev resting HR                 median r=-0.079  significant in 13/43 patients
  Prev sleep duration             median r=-0.026  significant in 11/43 patients
  Prev sleep fragmentation        median r=+0.021  significant in 5/43 patients


In [46]:
# ── Paired hourly time-series: good vs poor sleep night → next-day step profile
# Join high/low sleep flags from daily onto hourly df by (id, date)
_daily_flags = daily[['id', 'date', 'is_good_sleep', 'is_poor_sleep']].copy()
_daily_flags['date'] = pd.to_datetime(_daily_flags['date'])

_hourly = df[['id', 'time', 'steps']].copy()
_hourly['date'] = _hourly['time'].dt.normalize()
_hourly['hour'] = _hourly['time'].dt.hour
_hourly = _hourly.merge(_daily_flags, on=['id', 'date'], how='left')

# Focus on daytime hours 6–22 (active period)
_active = _hourly[_hourly['hour'].between(6, 22)]

good_profile = _active[_active['is_good_sleep'] == True].groupby('hour')['steps'].agg(['mean', 'std'])
poor_profile = _active[_active['is_poor_sleep'] == True].groupby('hour')['steps'].agg(['mean', 'std'])

hours = good_profile.index.tolist()

fig = go.Figure()

for label, profile, color in [
    ('Good sleep night (top 25%)', good_profile, '#4C72B0'),
    ('Poor sleep night (bottom 25%)', poor_profile, '#DD8452'),
]:
    _h = profile.index.tolist()
    _m = profile['mean'].values
    _s = profile['std'].fillna(0).values
    fig.add_trace(go.Scatter(
        x=_h, y=_m, mode='lines+markers',
        name=label, line=dict(color=color, width=2),
    ))
    fig.add_trace(go.Scatter(
        x=_h + _h[::-1],
        y=list(_m + _s) + list((_m - _s)[::-1]),
        fill='toself', fillcolor=color,
        opacity=0.15, line=dict(color='rgba(0,0,0,0)'),
        showlegend=False,
    ))

fig.update_layout(
    title='Direction 1: Hourly Step Profile Following Good vs Poor Sleep Night<br>'
          '<sup>Good = prev-night sleep in top 25% per patient · ribbon = ±1 SD</sup>',
    xaxis_title='Hour of day',
    yaxis_title='Mean hourly steps',
    xaxis=dict(tickmode='linear', dtick=2),
    height=420, width=800,
    legend=dict(orientation='h', yanchor='bottom', y=1.02),
)
fig.show()

## Direction 2: Activity → Next-Night Recovery

Do high-activity days predict elevated resting HR, shorter sleep, or more fragmented sleep the following night?

In [37]:
# ── Scatter: today's steps_z vs each next-night outcome ───────────────────────
# daily_next: daily_steps = today's activity; sum_sleep_minute/resting_hr = following night
D2_OUTCOMES = [
    ('resting_hr_z',              'Next-night resting HR (z)',     'High steps → elevated recovery HR?'),
    ('sum_sleep_minute_z',        'Next-night sleep duration (z)', 'High steps → more/less sleep?'),
    ('sleep_fragmentation_min_z', 'Next-night fragmentation (z)',  'High steps → more/less restless?'),
]

_plot_df = daily_next.dropna(subset=['daily_steps_z'] + [m[0] for m in D2_OUTCOMES])

fig = make_subplots(rows=1, cols=3, subplot_titles=[m[2] for m in D2_OUTCOMES], shared_xaxes=True)

for col_idx, (y_col, y_label, _) in enumerate(D2_OUTCOMES, start=1):
    for disease in DISEASE_ORDER:
        sub = _plot_df[_plot_df['disease_type'] == disease]
        fig.add_trace(go.Scatter(
            x=sub['daily_steps_z'], y=sub[y_col],
            mode='markers',
            marker=dict(color=DISEASE_COLORS[disease], size=4, opacity=0.45),
            name=disease, legendgroup=disease, showlegend=(col_idx == 1),
        ), row=1, col=col_idx)
        _x = sub['daily_steps_z'].dropna()
        _y = sub.loc[_x.index, y_col].dropna()
        _x = _x.loc[_y.index]
        if len(_x) > 5:
            _m, _b = np.polyfit(_x, _y, 1)
            _xr = np.array([_x.min(), _x.max()])
            fig.add_trace(go.Scatter(
                x=_xr, y=_m * _xr + _b, mode='lines',
                line=dict(color=DISEASE_COLORS[disease], width=2), showlegend=False,
            ), row=1, col=col_idx)
    fig.update_yaxes(title_text=y_label, row=1, col=col_idx)
    fig.update_xaxes(title_text="Today's steps (z)", row=1, col=col_idx)

fig.update_layout(
    title="Direction 2: Today's Steps → Next-Night Recovery<br>"
          '<sup>Within-patient z-scores · OLS trend per disease stage</sup>',
    height=480, width=1100,
    legend=dict(orientation='h', yanchor='bottom', y=1.08),
)
fig.show()

In [42]:
# ── Spearman r: today's steps vs next-night outcomes, by disease stage ─────────
# Uses within-patient z-scores so pooled ranking reflects deviation from each
# patient's own baseline, not between-patient differences in absolute level.
D2_NEXT_COLS   = ['resting_hr_z',            'sum_sleep_minute_z',   'sleep_fragmentation_min_z']
D2_NEXT_LABELS = ['Next resting HR (z)', 'Next sleep (min) (z)', 'Next sleep frag. (z)']

records = []
for stage in DISEASE_ORDER:
    sub = daily_next[daily_next['disease_type'] == stage]
    for col, label in zip(D2_NEXT_COLS, D2_NEXT_LABELS):
        valid = sub[['daily_steps_z', col]].dropna()
        r, p  = spearmanr(valid['daily_steps_z'], valid[col]) if len(valid) >= 10 else (np.nan, np.nan)
        records.append({'disease_type': stage, 'outcome': label, 'r': r, 'p': p, 'n': len(valid)})

corr_stage = pd.DataFrame(records)
corr_stage['sig'] = corr_stage['p'].apply(
    lambda p: '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
)

print(f'{"Disease Stage":<30}  {"Outcome":<25}  {"r":>6}  {"p":>8}  {"":>4}  {"n":>5}')
print('-' * 83)
for _, row in corr_stage.iterrows():
    print(f'{str(row["disease_type"]):<30}  {row["outcome"]:<25}  '
          f'{row["r"]:>+6.3f}  {row["p"]:>8.4f}  {row["sig"]:>4}  {row["n"]:>5}')

# ── Heatmap ───────────────────────────────────────────────────────────────────
r_pivot   = corr_stage.pivot(index='disease_type', columns='outcome', values='r').loc[DISEASE_ORDER, D2_NEXT_LABELS]
sig_pivot = corr_stage.pivot(index='disease_type', columns='outcome', values='sig').loc[DISEASE_ORDER, D2_NEXT_LABELS]
text_matrix = [
    [f'{r:+.2f} {s}' for r, s in zip(r_row, s_row)]
    for r_row, s_row in zip(r_pivot.values, sig_pivot.values)
]

fig = px.imshow(
    r_pivot.values,
    x=D2_NEXT_LABELS, y=DISEASE_ORDER,
    color_continuous_scale='RdBu_r',
    color_continuous_midpoint=0, range_color=[-0.5, 0.5],
    title='Direction 2: Steps → Next-Night Recovery by Disease Stage<br>'
          '<sup>Spearman r on within-patient z-scores · * p<0.05, ** p<0.01, *** p<0.001</sup>',
    aspect='auto', height=280, width=680,
)
for i, row in enumerate(text_matrix):
    for j, txt in enumerate(row):
        fig.add_annotation(x=j, y=i, text=txt, showarrow=False, font=dict(size=12, color='black'))
fig.update_layout(coloraxis_colorbar=dict(title='Spearman r'))
fig.show()

Disease Stage                   Outcome                         r         p            n
-----------------------------------------------------------------------------------
Early Disease Stage             Next resting HR (z)        +0.007    0.8591    ns    625
Early Disease Stage             Next sleep (min) (z)       -0.015    0.7015    ns    625
Early Disease Stage             Next sleep frag. (z)       +0.044    0.2770    ns    625
Fast Disease Progression        Next resting HR (z)        +0.122    0.0410     *    279
Fast Disease Progression        Next sleep (min) (z)       -0.098    0.1039    ns    279
Fast Disease Progression        Next sleep frag. (z)       -0.062    0.3045    ns    279
Late Disease Stage              Next resting HR (z)        +0.039    0.2607    ns    829
Late Disease Stage              Next sleep (min) (z)       -0.043    0.2206    ns    829
Late Disease Stage              Next sleep frag. (z)       -0.018    0.6108    ns    829


In [39]:
# ── Violin: activity quartile → each next-night outcome ───────────────────────
_plot_df = daily_next.dropna(subset=['steps_q', 'disease_type'])
_plot_df['steps_q'] = _plot_df['steps_q'].astype(str)
STEPS_Q_ORDER = ['Q1 (low)', 'Q2', 'Q3', 'Q4 (high)']

for y_col, y_label, title_note in D2_OUTCOMES:
    sub = _plot_df.dropna(subset=[y_col])
    fig = px.violin(
        sub,
        x='steps_q', y=y_col,
        color='disease_type',
        category_orders={'steps_q': STEPS_Q_ORDER, 'disease_type': DISEASE_ORDER},
        color_discrete_map=DISEASE_COLORS,
        box=True, points=False,
        title=f'Direction 2: Activity Quartile → {y_label}<br>'
              f'<sup>{title_note} · within-patient quartiles</sup>',
        labels={'steps_q': "Today's steps quartile (within-patient)", y_col: y_label},
        height=480, width=900,
    )
    fig.update_layout(violinmode='group')
    fig.show()

In [50]:
# ── Per-patient Spearman r: daily_steps_z → next-night outcomes ───────────────
D2_PAIRS = [
    ('resting_hr',            'Next resting\nHR'),
    ('sum_sleep_minute',      'Next sleep\nduration'),
    ('sleep_fragmentation_min', 'Next sleep\nfragmentation'),
]

rows_d2 = []
for pid in sorted(daily_next['id'].unique()):
    pat     = daily_next[daily_next['id'] == pid]
    disease = pat['disease_type'].iloc[0]
    row     = {'id': pid, 'disease_type': disease}
    for src_col, label in D2_PAIRS:
        sub = pat[['daily_steps_z', src_col]].dropna()
        if len(sub) >= 10:
            r, p = spearmanr(sub['daily_steps_z'], sub[src_col])
        else:
            r, p = np.nan, np.nan
        row[f'r_{src_col}']   = r
        row[f'sig_{src_col}'] = '*' if (not np.isnan(p) and p < 0.05) else ''
    rows_d2.append(row)

corr_d2 = pd.DataFrame(rows_d2)
corr_d2['disease_ord'] = corr_d2['disease_type'].map(
    {d: i for i, d in enumerate(DISEASE_ORDER)}
)
corr_d2 = corr_d2.sort_values(['disease_ord', 'r_resting_hr'], ascending=[True, False])

r_cols     = [f'r_{c}' for c, _ in D2_PAIRS]
labels     = [l for _, l in D2_PAIRS]
r_matrix   = corr_d2[r_cols].values
sig_matrix = corr_d2[[f'sig_{c}' for c, _ in D2_PAIRS]].values
y_labels   = [f"{row['id']} ({str(row['disease_type'])[:5]})" for _, row in corr_d2.iterrows()]

fig = px.imshow(
    r_matrix, x=labels, y=y_labels,
    color_continuous_scale='RdBu_r',
    color_continuous_midpoint=0, range_color=[-1, 1],
    title='Direction 2: Per-Patient Spearman r (Steps → Next-Night Recovery)<br>'
          '<sup>* p < 0.05 · sorted by disease stage · red = positive, blue = negative</sup>',
    aspect='auto', height=max(500, len(corr_d2) * 20), width=550,
)
for i in range(r_matrix.shape[0]):
    for j in range(r_matrix.shape[1]):
        if sig_matrix[i, j] == '*':
            fig.add_annotation(x=j, y=i, text='*', showarrow=False, font=dict(size=14, color='black'))
fig.update_layout(coloraxis_colorbar=dict(title='Spearman r'))
fig.update_yaxes(tickmode='linear', dtick=1, tickfont=dict(size=9))
fig.show()

print('\nMedian r across patients:')
for src_col, label in D2_PAIRS:
    med   = corr_d2[f'r_{src_col}'].median()
    n_sig = (corr_d2[f'sig_{src_col}'] == '*').sum()
    print(f'  {label.replace(chr(10)," "):30s}  median r={med:+.3f}  significant in {n_sig}/{len(corr_d2)} patients')


Median r across patients:
  Next resting HR                 median r=+0.065  significant in 12/42 patients
  Next sleep duration             median r=-0.012  significant in 7/42 patients
  Next sleep fragmentation        median r=-0.006  significant in 3/42 patients


In [41]:
# ── Paired hourly time-series: high vs low step days → next night's HR ────────
# Align daytime (same day, hours 6-23) + next morning (date+1, hours 0-8)

_daily_flags2 = daily_next[['id', 'date', 'is_high_step', 'is_low_step']].copy()
_daily_flags2['date'] = pd.to_datetime(_daily_flags2['date'])

_hourly = df[['id', 'time', 'steps', 'heartrate']].copy()
_hourly['date'] = _hourly['time'].dt.normalize()
_hourly['hour'] = _hourly['time'].dt.hour

# Day portion: join flags via same date
_day = _hourly[_hourly['hour'].between(6, 23)].merge(
    _daily_flags2, on=['id', 'date'], how='inner'
)
_day['rel_hour'] = _day['hour']  # 6–23

# Night portion: join flags via date-1 (the activity day before this night)
_night_flags = _daily_flags2.copy()
_night_flags['date'] = _night_flags['date'] + pd.Timedelta(days=1)
_night = _hourly[_hourly['hour'] <= 8].merge(
    _night_flags, on=['id', 'date'], how='inner'
)
_night['rel_hour'] = _night['hour'] + 24  # 24–32

_combined = pd.concat([_day, _night], ignore_index=True)

high_hr = _combined[_combined['is_high_step']].groupby('rel_hour')['heartrate'].agg(['mean', 'std'])
low_hr  = _combined[_combined['is_low_step']].groupby('rel_hour')['heartrate'].agg(['mean', 'std'])

fig = go.Figure()
for label, profile, color in [
    ('High-step day (top 25%)',   high_hr, '#C44E52'),
    ('Low-step day (bottom 25%)', low_hr,  '#4C72B0'),
]:
    _h = profile.index.tolist()
    _m = profile['mean'].values
    _s = profile['std'].fillna(0).values
    fig.add_trace(go.Scatter(x=_h, y=_m, mode='lines+markers',
                             name=label, line=dict(color=color, width=2)))
    fig.add_trace(go.Scatter(
        x=_h + _h[::-1], y=list(_m + _s) + list((_m - _s)[::-1]),
        fill='toself', fillcolor=color, opacity=0.15,
        line=dict(color='rgba(0,0,0,0)'), showlegend=False,
    ))

fig.add_vline(x=24, line_dash='dash', line_color='gray', annotation_text='midnight')
fig.add_vline(x=6,  line_dash='dot',  line_color='lightgray', annotation_text='6am')
fig.update_layout(
    title='Direction 2: Hourly HR Profile on High vs Low Step Days + Following Night<br>'
          '<sup>High/low = top/bottom 25% of steps per patient · ribbon = ±1 SD</sup>',
    xaxis=dict(
        title='Relative hour (6=day start, 24=midnight, 32=8am next day)',
        tickmode='linear', dtick=2,
        ticktext=[f'{h}:00' if h <= 23 else f'+{h-24}:00' for h in range(6, 33, 2)],
        tickvals=list(range(6, 33, 2)),
    ),
    yaxis_title='Mean hourly heart rate (bpm)',
    height=450, width=900,
    legend=dict(orientation='h', yanchor='bottom', y=1.02),
)
fig.show()